In [15]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)

# Diffusion coefficient problem 

We implement the inverse problem to recover the diffusion coefficient given the solution to the PDE. formally that is we consider the problem 
$$
\nabla a \nabla u = f \text{ in } \Omega
$$
$$
u = g \text{ on } \partial \Omega.
$$

As variational formulation this is given by the bilinear form

$$     b(u,v) = \int_\Omega a \nabla(u) \nabla(v) dx $$

and the linear form

$$    F(v) = \int_{\partial \Omega} f v ds. $$
As the bilinear form depends linearly on the coefficient $a$ we may use the provided class `SecondOrderEllipticCoefficientPDE`.


In [136]:
from regpy.operators.ngsolve import SecondOrderEllipticCoefficientPDE
import ngsolve as ngs

class diffusion(SecondOrderEllipticCoefficientPDE):
    def __init__(self, domain, sol_domain,bdr_val = None,a_bdr_val=None):
        super().__init__(domain, sol_domain, bdr_val=bdr_val,a_bdr_val=a_bdr_val)

    def _bf(self,a,u,v):
        return a*ngs.grad(u)*ngs.grad(v)*ngs.dx
    
    def _lf(self):
        p = ngs.GridFunction(self.codomain.fes)
        p.Set(-2*ngs.exp(ngs.x+ngs.y))
        lf = ngs.LinearForm(self.codomain.fes)
        lf += p * self.v * ngs.dx
        return lf.Assemble()

Now we define a mesh and FES spaces for the coefficients and solution spaces and construct the operator by giving it the boundary values. Moreover define the exact solution and take their boundary values as input to the operator. 

In [137]:
from netgen.geom2d import unit_square
from regpy.vecsps.ngsolve import NgsSpace

bdr = "left|top|right|bottom"
mesh = ngs.Mesh(unit_square.GenerateMesh(maxh=0.05))
fes_domain = ngs.H1(mesh, order=6, dirichlet = bdr)
domain = NgsSpace(fes_domain,bdr = bdr)

bdr = "left|top|right|bottom"
fes_codomain = ngs.H1(mesh, order=6, dirichlet=bdr)
codomain = NgsSpace(fes_codomain, bdr=bdr)

bdr_coeff = ngs.exp(ngs.x+ngs.y)
bdr_gf = ngs.GridFunction(codomain.fes)
bdr_gf.Set(bdr_coeff,definedon=codomain.fes.mesh.Boundaries(codomain.bdr))
bdr_val = codomain.from_ngs(bdr_gf)

exact_solution_coeff = 1+0.2*ngs.exp(-2*(ngs.x-0.5)**2-2*(ngs.y-0.5)**2)
exact_solution = domain.from_ngs( exact_solution_coeff )
p = ngs.GridFunction(domain.fes)
p.Set(exact_solution_coeff,definedon=domain.fes.mesh.Boundaries(domain.bdr))
a_bdr_val = domain.from_ngs( p )

op = diffusion(
    domain, codomain, bdr_val=bdr_val,a_bdr_val = a_bdr_val
)

Get data by perturbing the exact data by some random noise. 

In [138]:
exact_data = op(exact_solution)
noise = 0.00005 * codomain.randn()
data = exact_data+noise

Define a an guess by choosing the constant one function on the domain. Then define a regularization setting by choosing appropriate norms on both the domain and codomain. We choose Landweber as our regularization scheme. Combined with a stoping rule that is composed of a max iteration and relative change in the data.  

In [151]:
from regpy.solvers import RegularizationSetting
from regpy.solvers.nonlinear.landweber import Landweber
import regpy.stoprules as rules
from regpy.hilbert import Hm0,Sobolev

init = domain.ones()

setting = RegularizationSetting(op=op, penalty=Hm0, data_fid=Hm0)


landweber = Landweber(setting, data, init,op_norm_method="power_method")

stoprule = (
        rules.CountIterations(5000) +
        rules.RelativeChangeData(setting.h_codomain.norm,data,0.000001) 
)

Do the inversion by calling `landweber.run(stoprule)`. 

In [152]:
from ngsolve.webgui import Draw

reco, reco_data = landweber.run(stoprule)

Draw(exact_solution_coeff, op.domain.fes.mesh, "exact")

# Draw reconstructed solution
Draw(domain.to_ngs(reco),op.domain.fes.mesh, "reconstruction")

# Draw data space
Draw(codomain.to_ngs(data),op.codomain.fes.mesh, "exact data")
Draw(codomain.to_ngs(reco_data),op.codomain.fes.mesh, "data of reconstruction")

2024-06-27 12:00:26,655 INFO CountIterations      :: iteration = 1 / 5000
2024-06-27 12:00:26,657 INFO RelativeChangeData   :: RelativeChangeData = 0.1532620387386496, cutoff = 1e-06
2024-06-27 12:00:26,880 INFO Landweber            :: |residual| = 0.1532620387386496
2024-06-27 12:00:26,880 INFO CountIterations      :: iteration = 2 / 5000
2024-06-27 12:00:26,884 INFO RelativeChangeData   :: RelativeChangeData = 0.09779157312076864, cutoff = 1e-06
2024-06-27 12:00:27,075 INFO Landweber            :: |residual| = 0.08820126891647863
2024-06-27 12:00:27,075 INFO CountIterations      :: iteration = 3 / 5000
2024-06-27 12:00:27,078 INFO RelativeChangeData   :: RelativeChangeData = 0.021034683710184774, cutoff = 1e-06
2024-06-27 12:00:27,304 INFO Landweber            :: |residual| = 0.07475791866616681
2024-06-27 12:00:27,304 INFO CountIterations      :: iteration = 4 / 5000
2024-06-27 12:00:27,312 INFO RelativeChangeData   :: RelativeChangeData = 0.011971413284995453, cutoff = 1e-06
2024-0

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene